# 01 — Exploratory Data Analysis: S&P 500 regimes

Before running this notebook, run the pipeline from the repo root:

```bash
python -m sp500.data_download
python -m sp500.data_prep
python -m sp500.features
```

Write the key findings in `docs/04_eda.md`.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.graphics.tsaplots import plot_acf

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from sp500.config import DATA_PROCESSED, FIGURES, REPORTS
from sp500.labels import episodes

FIGURES.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 4)

def save(fig, name):
    fig.savefig(FIGURES / f'{name}.png', dpi=150, bbox_inches='tight')

In [ ]:
daily = pd.read_parquet(DATA_PROCESSED / 'sp500_daily.parquet')
table = pd.read_parquet(DATA_PROCESSED / 'model_table.parquet')
print('daily:', daily.shape, ' model table:', table.shape)
print(daily.index.min().date(), '->', daily.index.max().date())
daily.info()

## 1. Structure and quality

In [ ]:
pd.read_csv(REPORTS / 'data_quality.csv')

In [ ]:
# When does each column become available?
availability = pd.DataFrame({
    'first_valid': daily.apply(lambda s: s.first_valid_index()),
    'last_valid': daily.apply(lambda s: s.last_valid_index()),
    'missing_pct': daily.isna().mean().mul(100).round(1),
})
availability

In [ ]:
# Share of missing values per column and year
missing_by_year = daily.isna().groupby(daily.index.year).mean()
fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(missing_by_year.T, cmap='Greys', cbar_kws={'label': 'share missing'}, ax=ax)
ax.set_title('Missing data by year')
save(fig, 'missing_by_year')

In [ ]:
# Largest daily moves: genuine events (1987, 2008, 2020) or data errors?
r = np.log(daily['gspc_close']).diff().rename('log_ret')
r[r.abs() > 0.07].sort_values().to_frame()

## 2. Price and returns

In [ ]:
fig, ax = plt.subplots()
daily['gspc_close'].plot(ax=ax, logy=True, lw=0.8)
ax.set_title('S&P 500 price index (log scale)')
ax.set_xlabel('')
save(fig, 'gspc_log_price')

In [ ]:
ret = r.dropna()
print(f'mean (ann.) {ret.mean()*252:.2%}   vol (ann.) {ret.std()*np.sqrt(252):.2%}')
print(f'skew {stats.skew(ret):.2f}   excess kurtosis {stats.kurtosis(ret):.2f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(ret, bins=200, stat='density', ax=axes[0])
x = np.linspace(ret.min(), ret.max(), 500)
axes[0].plot(x, stats.norm.pdf(x, ret.mean(), ret.std()), color='C3', label='normal')
axes[0].set_xlim(-0.08, 0.08)
axes[0].legend()
axes[0].set_title('Daily log returns vs. normal')
stats.probplot(ret, dist='norm', plot=axes[1])
axes[1].set_title('Q-Q plot')
save(fig, 'returns_distribution')

In [ ]:
# Returns are close to uncorrelated; absolute returns are not (volatility clustering)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
plot_acf(ret, lags=60, ax=axes[0], title='ACF of daily returns')
plot_acf(ret.abs(), lags=60, ax=axes[1], title='ACF of |daily returns|')
save(fig, 'acf_returns')

In [ ]:
decade = (ret.index.year // 10) * 10
ret.groupby(decade).agg(
    days='count',
    ann_return=lambda s: s.mean() * 252,
    ann_vol=lambda s: s.std() * np.sqrt(252),
    worst_day='min',
    best_day='max',
).round(3)

## 3. Volatility

In [ ]:
fig, ax = plt.subplots()
table['rv_21d'].plot(ax=ax, lw=0.6, label='21d realized vol')
table['stress_threshold'].plot(ax=ax, color='C3', label='expanding 80th pct (past only)')
ax.set_title('Realized volatility and stress threshold')
ax.legend()
save(fig, 'realized_vol_threshold')

In [ ]:
vol_cols = [c for c in ['rv_21d', 'parkinson_21d', 'garman_klass_21d'] if c in table]
fig, ax = plt.subplots()
table.loc['2007':'2010', vol_cols].plot(ax=ax, lw=0.8)
ax.set_title('Volatility estimators, 2007-2010')
save(fig, 'vol_estimators')
table[vol_cols].corr().round(2)

## 4. Regimes

In [ ]:
def shade(ax, label, color, name, alpha=0.25):
    for i, row in episodes(label).iterrows():
        ax.axvspan(row['start'], row['end'], color=color, alpha=alpha, lw=0, label=name if i == 0 else None)

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
for ax, col, color, title in [
    (axes[0], 'bear', 'C3', 'Bear (peak to trough, hindsight)'),
    (axes[1], 'bear_recovery', 'C1', 'Bear (20% down until recovery, causal)'),
    (axes[2], 'stressed', 'C4', 'Stressed (21d vol > past 80th pct)'),
]:
    daily['gspc_close'].plot(ax=ax, logy=True, color='k', lw=0.7)
    shade(ax, table[col], color, title)
    ax.set_title(title)
    ax.set_xlabel('')
save(fig, 'regimes_on_price')

In [ ]:
# Sanity check: are the known bear markets captured?
known = {
    'Dot-com 2000-02': ('2000-03-24', '2002-10-09'),
    'GFC 2007-09': ('2007-10-09', '2009-03-09'),
    'COVID 2020': ('2020-02-19', '2020-03-23'),
    'Inflation 2022': ('2022-01-03', '2022-10-12'),
}
pd.DataFrame({
    name: {
        'bear_share': table.loc[a:b, 'bear'].mean(),
        'bear_recovery_share': table.loc[a:b, 'bear_recovery'].mean(),
        'stressed_share': table.loc[a:b, 'stressed'].mean(),
    }
    for name, (a, b) in known.items()
}).T.round(2)

In [ ]:
# Class balance and episode lengths
summary = {}
for col in ['bear', 'bear_recovery', 'stressed']:
    ep = episodes(table[col])
    summary[col] = {
        'share_of_days': table[col].mean(),
        'episodes': len(ep),
        'median_days': ep['days'].median(),
        'max_days': ep['days'].max(),
    }
pd.DataFrame(summary).T.round(3)

In [ ]:
# How often do bear and stressed overlap?
pd.crosstab(table['bear'], table['stressed'], normalize='all').round(3)

In [ ]:
# Return and volatility by regime
for col in ['bear', 'stressed']:
    print(table.groupby(col)['ret_1d'].agg(
        ann_return=lambda s: s.mean() * 252,
        ann_vol=lambda s: s.std() * np.sqrt(252),
        days='count',
    ).round(3), '\n')

## 5. Features vs. target

In [ ]:
feature_cols = [c for c in [
    'ret_1d', 'mom_1m', 'mom_3m', 'mom_6m', 'mom_12m', 'dist_ma50', 'dist_ma200', 'drawdown',
    'rv_21d', 'rv_63d', 'parkinson_21d', 'garman_klass_21d', 'gap_open', 'spy_vol_ratio_20d', 'es_ret_1d',
] if c in table]

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(table[feature_cols].corr(), cmap='RdBu_r', center=0, annot=True, fmt='.2f', ax=ax)
ax.set_title('Feature correlations')
save(fig, 'feature_correlations')

In [ ]:
target = 'target_stressed_h21'
plot_cols = ['mom_3m', 'dist_ma200', 'drawdown', 'rv_21d']
d = table[plot_cols + [target]].dropna()
fig, axes = plt.subplots(1, len(plot_cols), figsize=(16, 4))
for ax, col in zip(axes, plot_cols):
    sns.boxplot(data=d, x=target, y=col, ax=ax, showfliers=False)
    ax.set_title(col)
fig.suptitle(f'Features today vs. {target}')
save(fig, 'features_by_target')

In [ ]:
# Persistence: P(regime at t+h | regime at t). High values = strong 'no change' baseline
for label in ['bear', 'stressed']:
    for h in (5, 21):
        ct = pd.crosstab(table[label], table[f'target_{label}_h{h}'], normalize='index').round(3)
        print(f'{label}, h={h}\n{ct}\n')

## Findings

Summarize the main findings here and in `docs/04_eda.md`.